In [ ]:
# Example 1: ChatPerplexity is exposed as a top-level export.
import langchain_perplexity
from langchain_perplexity import ChatPerplexity

assert hasattr(langchain_perplexity, 'ChatPerplexity'), 'ChatPerplexity should be a top-level export'
assert ChatPerplexity is langchain_perplexity.ChatPerplexity
print(f"ChatPerplexity class: {ChatPerplexity}")
print(f"Module: {ChatPerplexity.__module__}")

In [ ]:
# Example 2: ChatPerplexity inherits from BaseChatModel and exposes Pydantic v2 fields.
# (model_json_schema() raises PydanticInvalidForJsonSchema with pydantic 2.12 on the
# Callable-typed custom_get_token_ids field, so introspect model_fields instead.)
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_perplexity import ChatPerplexity

assert issubclass(ChatPerplexity, BaseChatModel), 'ChatPerplexity must subclass BaseChatModel'

fields = ChatPerplexity.model_fields
assert isinstance(fields, dict) and fields
print(f"Number of model fields: {len(fields)}")
# Common LangChain chat-model config fields
for name in ('model', 'temperature', 'max_tokens'):
    print(f"  has field {name!r}: {name in fields}")
    assert name in fields

In [ ]:
# Example 3: Instantiate ChatPerplexity with a dummy api key; check attributes.
from pydantic import SecretStr
from langchain_perplexity import ChatPerplexity

chat = ChatPerplexity(
    model='sonar',
    pplx_api_key=SecretStr('dummy-test-key-not-real'),
    temperature=0.0,
)
assert chat.model == 'sonar'
assert chat.temperature == 0.0
# _llm_type is a LangChain convention identifying the integration.
llm_type = chat._llm_type
print(f"_llm_type = {llm_type!r}")
assert isinstance(llm_type, str) and len(llm_type) > 0

In [ ]:
# Example 4: Runnable protocol surface + LangSmith tracing params.
# (_identifying_params returns {} on langchain-perplexity 1.4.0 — the model
# identity now surfaces through the standard _get_ls_params tracing hook.)
from pydantic import SecretStr
from langchain_core.runnables import Runnable
from langchain_perplexity import ChatPerplexity

chat = ChatPerplexity(
    model='sonar',
    pplx_api_key=SecretStr('dummy'),
)
# Chat models are Runnables (invoke/batch/stream/ainvoke).
assert isinstance(chat, Runnable)
for method in ('invoke', 'batch', 'stream', 'ainvoke', 'astream'):
    assert callable(getattr(chat, method)), f'missing Runnable method {method!r}'

ls_params = chat._get_ls_params()
print(f"ls params: {ls_params}")
assert ls_params.get('ls_provider') == 'perplexity'
assert ls_params.get('ls_model_name') == 'sonar'

In [ ]:
# Example 5: bind_tools / structured-output methods exist on the chat model (LangChain v0.3+ surface).
from pydantic import SecretStr
from langchain_perplexity import ChatPerplexity

chat = ChatPerplexity(model='sonar', pplx_api_key=SecretStr('dummy'))

# These methods are part of the standard BaseChatModel API. They may or may not be
# fully implemented for Perplexity, but should at minimum exist as attributes.
for method_name in ('bind', 'with_config', 'with_retry', 'with_fallbacks'):
    assert callable(getattr(chat, method_name)), f'expected {method_name} on ChatPerplexity'

# Build a no-op bound runnable to confirm .bind works end-to-end.
bound = chat.bind(stop=['\n'])
assert bound is not None
print(f"bound runnable: {type(bound).__name__}")
print('langchain_perplexity smoke tests passed')